In [11]:
import pandas as pd
import numpy as np

df = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet')

cutoff = df['TransactionDT'].quantile(0.8)
train = df[df['TransactionDT'] <= cutoff].copy()
test = df[df['TransactionDT'] > cutoff].copy()

print("train:", train.shape, "fraud rate:", train['isFraud'].mean())
print("test:", test.shape, "fraud rate:", test['isFraud'].mean())

train: (472432, 23) fraud rate: 0.03513521522674162
test: (118108, 23) fraud rate: 0.034409184813899145


In [12]:
import os
print(os.path.exists('data'))
print(os.listdir('data') if os.path.exists('data') else "data/ folder is gone")

False
data/ folder is gone


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_dir = '/content/drive/MyDrive/ringsentinel'
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f'{project_dir}/data', exist_ok=True)
os.makedirs(f'{project_dir}/reports', exist_ok=True)

Mounted at /content/drive


In [50]:

from google.colab import userdata
token = userdata.get('KAGGLE_API_TOKEN').strip()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)

!kaggle competitions download -c ieee-fraud-detection
!unzip -q ieee-fraud-detection.zip -d data/

import pandas as pd, numpy as np

train_tx = pd.read_csv('data/train_transaction.csv')
train_id = pd.read_csv('data/train_identity.csv')
df = train_tx.merge(train_id, on='TransactionID', how='left')

def downcast(df):
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df
df = downcast(df)
del train_tx, train_id

baseline_cols = [
    'TransactionAmt', 'ProductCD', 'card1','card2','card3','card4','card5','card6',
    'addr1','addr2', 'P_emaildomain','R_emaildomain', 'DeviceType',
    'C1','C2','C13','C14',
    'D1','D4','D10',
]
df[baseline_cols + ['isFraud','TransactionDT','TransactionID']].to_parquet(
    f'{project_dir}/data/baseline_features.parquet'
)
print("saved to Drive.")

ieee-fraud-detection.zip: Skipping, found more recently modified local copy (use --force to force download)
replace data/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
A
A
A


KeyboardInterrupt: 

In [14]:
cat_cols = ['ProductCD','card4','card6','P_emaildomain','R_emaildomain','DeviceType']

df = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet')

cutoff = df['TransactionDT'].quantile(0.8)
train = df[df['TransactionDT'] <= cutoff].copy()
test = df[df['TransactionDT'] > cutoff].copy()

for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = pd.Categorical(test[col], categories=train[col].cat.categories)

In [15]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve

feature_cols = [c for c in train.columns if c not in ['isFraud','TransactionDT','TransactionID']]

train_set = lgb.Dataset(train[feature_cols], label=train['isFraud'], categorical_feature=cat_cols)
test_set = lgb.Dataset(test[feature_cols], label=test['isFraud'], categorical_feature=cat_cols, reference=train_set)

params = {
    'objective': 'binary',
    'metric': 'average_precision',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'verbose': -1
}

model_a = lgb.train(
    params, train_set,
    num_boost_round=500,
    valid_sets=[test_set],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)

Training until validation scores don't improve for 30 rounds
[50]	valid_0's average_precision: 0.467697
[100]	valid_0's average_precision: 0.486264
[150]	valid_0's average_precision: 0.495935
[200]	valid_0's average_precision: 0.500771
[250]	valid_0's average_precision: 0.504903
[300]	valid_0's average_precision: 0.507902
[350]	valid_0's average_precision: 0.509804
[400]	valid_0's average_precision: 0.512144
[450]	valid_0's average_precision: 0.514925
[500]	valid_0's average_precision: 0.516444
Did not meet early stopping. Best iteration is:
[491]	valid_0's average_precision: 0.516582


In [16]:
preds = model_a.predict(test[feature_cols])
pr_auc = average_precision_score(test['isFraud'], preds)
print(f"Model A PR-AUC: {pr_auc:.4f}")

precision, recall, thresholds = precision_recall_curve(test['isFraud'], preds)

for t in [0.3, 0.5, 0.7, 0.9]:
    pred_labels = (preds >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    p = precision_score(test['isFraud'], pred_labels)
    r = recall_score(test['isFraud'], pred_labels)
    print(f"threshold={t}: precision={p:.3f}, recall={r:.3f}")

Model A PR-AUC: 0.5166
threshold=0.3: precision=0.664, recall=0.419
threshold=0.5: precision=0.793, recall=0.336
threshold=0.7: precision=0.855, recall=0.261
threshold=0.9: precision=0.886, recall=0.152


In [17]:
model_a.save_model(f'{project_dir}/reports/model_a_baseline.txt')
np.save(f'{project_dir}/reports/model_a_test_preds.npy', preds)
np.save(f'{project_dir}/reports/model_a_test_labels.npy', test['isFraud'].values)

In [18]:
import pandas as pd, numpy as np, networkx as nx

link_cols = ['card1', 'card2', 'card3', 'addr1']

df_link = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet',
                           columns=['TransactionID', 'TransactionDT', 'isFraud'] + link_cols)
print(df_link.shape)

(590540, 7)


In [19]:
G = nx.Graph()

tx_ids = df_link['TransactionID'].values
G.add_nodes_from((f't_{tid}' for tid in tx_ids), bipartite=0)

for col in link_cols:
    vals = df_link[col]
    mask = vals.notna()
    tx_nodes = [f't_{tid}' for tid in tx_ids[mask]]
    val_nodes = [f'{col}_{v}' for v in vals[mask]]
    G.add_edges_from(zip(tx_nodes, val_nodes))

print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

nodes: 605039 edges: 2285956


In [20]:
components = list(nx.connected_components(G))
print("total components:", len(components))

tx_to_ring = {}
for ring_id, comp in enumerate(components):
    for node in comp:
        if node.startswith('t_'):
            tx_id = int(node[2:])
            tx_to_ring[tx_id] = ring_id

df_link['ring_id'] = df_link['TransactionID'].map(tx_to_ring)

total components: 4


In [21]:
ring_sizes = df_link.groupby('ring_id')['TransactionID'].transform('count')
df_link['ring_size'] = ring_sizes

print(df_link['ring_size'].describe())
print(df_link['ring_size'].value_counts().sort_index().head(20))

count    590540.000000
mean     590534.000020
std        1331.010894
min           1.000000
25%      590537.000000
50%      590537.000000
75%      590537.000000
max      590537.000000
Name: ring_size, dtype: float64
ring_size
1              3
590537    590537
Name: count, dtype: int64


In [22]:
bins = [0, 1, 2, 5, 10, 50, np.inf]
labels = ['1 (isolated)', '2', '3-5', '6-10', '11-50', '50+']
df_link['ring_size_bucket'] = pd.cut(df_link['ring_size'], bins=bins, labels=labels)

fraud_by_ring_size = df_link.groupby('ring_size_bucket', observed=True)['isFraud'].agg(['mean', 'count'])
print(fraud_by_ring_size)

                     mean   count
ring_size_bucket                 
1 (isolated)      0.00000       3
50+               0.03499  590537


In [23]:
df_link[['TransactionID', 'ring_id', 'ring_size']].to_parquet(
    f'{project_dir}/data/ring_assignments.parquet'
)
print("saved.")

saved.


In [24]:
link_cols = ['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']

df_link = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet',
                           columns=['TransactionID', 'TransactionDT', 'isFraud'] + link_cols)

df_link['fingerprint'] = df_link[link_cols].astype(str).agg('_'.join, axis=1)

print("unique fingerprints:", df_link['fingerprint'].nunique(), "out of", len(df_link), "rows")

unique fingerprints: 42999 out of 590540 rows


In [25]:
G = nx.Graph()
tx_ids = df_link['TransactionID'].values
G.add_nodes_from((f't_{tid}' for tid in tx_ids), bipartite=0)

fps = df_link['fingerprint'].values
tx_nodes = [f't_{tid}' for tid in tx_ids]
fp_nodes = [f'fp_{fp}' for fp in fps]
G.add_edges_from(zip(tx_nodes, fp_nodes))

print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

nodes: 633539 edges: 590540


In [26]:
fp_node_degrees = pd.Series(dict(G.degree(fp_nodes)))
print(fp_node_degrees.describe())
print(fp_node_degrees.sort_values(ascending=False).head(10))

count    42999.000000
mean        13.733808
std        101.047622
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max       9900.000000
dtype: float64
fp_15885_545.0_185.0_138.0_nan_nan       9900
fp_17188_321.0_150.0_226.0_299.0_87.0    5862
fp_12695_490.0_150.0_226.0_325.0_87.0    5766
fp_9500_321.0_150.0_226.0_204.0_87.0     4647
fp_3154_408.0_185.0_224.0_nan_nan        4398
fp_12839_321.0_150.0_226.0_264.0_87.0    3538
fp_16132_111.0_150.0_226.0_299.0_87.0    3523
fp_15497_490.0_150.0_226.0_299.0_87.0    3419
fp_9500_321.0_150.0_226.0_272.0_87.0     2715
fp_5812_408.0_185.0_224.0_nan_nan        2639
dtype: int64


In [27]:

HUB_THRESHOLD = 100
hub_nodes = fp_node_degrees[fp_node_degrees > HUB_THRESHOLD].index.tolist()
print(f"pruning {len(hub_nodes)} hub nodes")
G.remove_nodes_from(hub_nodes)

pruning 847 hub nodes


In [28]:
components = list(nx.connected_components(G))
print("total components:", len(components))

tx_to_ring = {}
for ring_id, comp in enumerate(components):
    for node in comp:
        if node.startswith('t_'):
            tx_id = int(node[2:])
            tx_to_ring[tx_id] = ring_id

df_link['ring_id'] = df_link['TransactionID'].map(tx_to_ring)
df_link['ring_id'] = df_link['ring_id'].fillna(-1).astype(int)

total components: 350246


In [29]:
ring_sizes = df_link.groupby('ring_id')['TransactionID'].transform('count')
df_link['ring_size'] = ring_sizes

print(df_link['ring_size'].describe())

count    590540.000000
mean         14.807603
std          23.509642
min           1.000000
25%           1.000000
50%           1.000000
75%          20.000000
max         100.000000
Name: ring_size, dtype: float64


In [30]:
bins = [0, 1, 2, 5, 10, 50, np.inf]
labels = ['1 (isolated)', '2', '3-5', '6-10', '11-50', '50+']
df_link['ring_size_bucket'] = pd.cut(df_link['ring_size'], bins=bins, labels=labels)

fraud_by_ring_size = df_link.groupby('ring_size_bucket', observed=True)['isFraud'].agg(['mean', 'count'])
print(fraud_by_ring_size)

                      mean   count
ring_size_bucket                  
1 (isolated)      0.040041  325464
2                 0.030977   12816
3-5               0.029960   27236
6-10              0.025919   33759
11-50             0.029243  128373
50+               0.028446   62892


In [31]:
df_link[['TransactionID', 'ring_id', 'ring_size']].to_parquet(
    f'{project_dir}/data/ring_assignments.parquet'
)
print("saved.")

saved.


In [34]:
device_cols = ['DeviceType', 'P_emaildomain']
df_dev = pd.read_parquet(
    f'{project_dir}/data/baseline_features.parquet',
    columns=['TransactionID', 'TransactionDT', 'isFraud'] + device_cols
)

In [36]:

df_dev_present = df_dev.dropna(subset=device_cols).copy()

df_dev_present['fingerprint'] = df_dev_present[device_cols].astype(str).agg('_'.join, axis=1)
print("unique fingerprints:", df_dev_present['fingerprint'].nunique(), "out of", len(df_dev_present))

unique fingerprints: 117 out of 127770


In [37]:
G2 = nx.Graph()
tx_ids = df_dev_present['TransactionID'].values
G2.add_nodes_from((f't_{tid}' for tid in tx_ids), bipartite=0)

fps = df_dev_present['fingerprint'].values
tx_nodes = [f't_{tid}' for tid in tx_ids]
fp_nodes = [f'fp_{fp}' for fp in fps]
G2.add_edges_from(zip(tx_nodes, fp_nodes))

fp_node_degrees = pd.Series(dict(G2.degree(set(fp_nodes))))
print(fp_node_degrees.describe())
print(fp_node_degrees.sort_values(ascending=False).head(10))

count      117.000000
mean      1092.051282
std       4021.004045
min          2.000000
25%         22.000000
50%         58.000000
75%        284.000000
max      28854.000000
dtype: float64
fp_desktop_gmail.com        28854
fp_mobile_gmail.com         24038
fp_desktop_anonymous.com    13611
fp_desktop_hotmail.com      13419
fp_mobile_hotmail.com       11873
fp_desktop_yahoo.com         6237
fp_mobile_yahoo.com          5156
fp_mobile_anonymous.com      3714
fp_desktop_aol.com           2721
fp_desktop_comcast.net       1551
dtype: int64


In [38]:
HUB_THRESHOLD = 100
hub_nodes = fp_node_degrees[fp_node_degrees > HUB_THRESHOLD].index.tolist()
print(f"pruning {len(hub_nodes)} hub nodes")
G2.remove_nodes_from(hub_nodes)

pruning 49 hub nodes


In [39]:
components2 = list(nx.connected_components(G2))
print("total components:", len(components2))

tx_to_ring2 = {}
for ring_id, comp in enumerate(components2):
    for node in comp:
        if node.startswith('t_'):
            tx_to_ring2[int(node[2:])] = ring_id

df_dev_present['ring_id'] = df_dev_present['TransactionID'].map(tx_to_ring2).fillna(-1).astype(int)
df_dev_present['ring_size'] = df_dev_present.groupby('ring_id')['TransactionID'].transform('count')

bins = [0, 1, 2, 5, 10, 50, np.inf]
labels = ['1 (isolated)', '2', '3-5', '6-10', '11-50', '50+']
df_dev_present['ring_size_bucket'] = pd.cut(df_dev_present['ring_size'], bins=bins, labels=labels)

print(df_dev_present.groupby('ring_size_bucket', observed=True)['isFraud'].agg(['mean', 'count']))

total components: 125800
                      mean   count
ring_size_bucket                  
1 (isolated)      0.081085  125732
2                 0.000000       2
3-5               0.034483      29
6-10              0.118644      59
11-50             0.039234    1096
50+               0.048122     852


In [40]:
check_cols = ['DeviceInfo', 'DeviceType', 'P_emaildomain', 'id_31', 'id_33']
df_check = pd.read_parquet(f'{project_dir}/data/baseline_features.parquet',
                            columns=['TransactionID'])

In [42]:

train_id = pd.read_csv('data/train_identity.csv', usecols=['DeviceInfo', 'id_31', 'id_33'])

for col in ['DeviceInfo', 'id_31', 'id_33']:
    print(col, "unique:", train_id[col].nunique(), "| top 5:")
    print(train_id[col].value_counts().head(5))
    print()

DeviceInfo unique: 1786 | top 5:
DeviceInfo
Windows        47722
iOS Device     19782
MacOS          12573
Trident/7.0     7440
rv:11.0         1901
Name: count, dtype: int64

id_31 unique: 130 | top 5:
id_31
chrome 63.0              22000
mobile safari 11.0       13423
mobile safari generic    11474
ie 11.0 for desktop       9030
safari generic            8195
Name: count, dtype: int64

id_33 unique: 260 | top 5:
id_33
1920x1080    16874
1366x768      8605
1334x750      6447
2208x1242     4900
1440x900      4384
Name: count, dtype: int64



In [44]:
import pandas as pd

train_id = pd.read_csv('data/train_identity.csv', usecols=['TransactionID', 'DeviceInfo', 'id_31', 'id_33'])
train_tx = pd.read_csv('data/train_transaction.csv', usecols=['TransactionID', 'TransactionDT', 'isFraud', 'P_emaildomain'])

df_dev = train_tx.merge(train_id, on='TransactionID', how='inner')

device_cols = ['DeviceInfo', 'id_31', 'id_33', 'P_emaildomain']
df_dev = df_dev[df_dev['DeviceInfo'].notna()].copy()

print("rows:", len(df_dev))
df_dev['fingerprint'] = df_dev[device_cols].astype(str).agg('_'.join, axis=1)
print("unique fingerprints:", df_dev['fingerprint'].nunique(), "out of", len(df_dev))

rows: 118666
unique fingerprints: 16222 out of 118666


In [45]:
G3 = nx.Graph()
tx_ids = df_dev['TransactionID'].values
G3.add_nodes_from((f't_{tid}' for tid in tx_ids), bipartite=0)

fps = df_dev['fingerprint'].values
tx_nodes = [f't_{tid}' for tid in tx_ids]
fp_nodes = [f'fp_{fp}' for fp in fps]
G3.add_edges_from(zip(tx_nodes, fp_nodes))

fp_node_degrees = pd.Series(dict(G3.degree(set(fp_nodes))))
print(fp_node_degrees.describe())
print(fp_node_degrees.sort_values(ascending=False).head(10))

count    16222.000000
mean         7.315128
std         48.275834
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       2595.000000
dtype: float64
fp_Windows_chrome 63.0_nan_gmail.com                      2595
fp_Windows_chrome 63.0_nan_hotmail.com                    2273
fp_iOS Device_mobile safari 11.0_1334x750_gmail.com       1636
fp_Windows_chrome 63.0_1920x1080_gmail.com                1399
fp_iOS Device_mobile safari 11.0_2208x1242_gmail.com      1371
fp_Windows_chrome 65.0_nan_gmail.com                      1036
fp_Windows_chrome 64.0_nan_gmail.com                      1023
fp_Windows_chrome 63.0_1920x1080_anonymous.com             971
fp_Windows_edge 16.0_1366x768_nan                          957
fp_iOS Device_mobile safari generic_1334x750_gmail.com     951
dtype: int64


In [46]:
threshold = fp_node_degrees.quantile(0.99)
print("99th percentile degree:", threshold)

hub_nodes = fp_node_degrees[fp_node_degrees > threshold].index.tolist()
print(f"pruning {len(hub_nodes)} hub nodes")
G3.remove_nodes_from(hub_nodes)

99th percentile degree: 102.0
pruning 162 hub nodes


In [47]:
components3 = list(nx.connected_components(G3))
print("total components:", len(components3))

tx_to_ring3 = {}
for ring_id, comp in enumerate(components3):
    for node in comp:
        if node.startswith('t_'):
            tx_to_ring3[int(node[2:])] = ring_id

df_dev['ring_id'] = df_dev['TransactionID'].map(tx_to_ring3).fillna(-1).astype(int)
df_dev['ring_size'] = df_dev.groupby('ring_id')['TransactionID'].transform('count')

bins = [0, 1, 2, 5, 10, 50, np.inf]
labels = ['1 (isolated)', '2', '3-5', '6-10', '11-50', '50+']
df_dev['ring_size_bucket'] = pd.cut(df_dev['ring_size'], bins=bins, labels=labels)

print(df_dev.groupby('ring_size_bucket', observed=True)['isFraud'].agg(['mean', 'count']))

total components: 67989
                      mean  count
ring_size_bucket                 
1 (isolated)      0.059309  60446
2                 0.052292   4800
3-5               0.090325   9798
6-10              0.101503   9251
11-50             0.090217  23920
50+               0.075495  10451


In [48]:
df_dev[['TransactionID', 'ring_id', 'ring_size']].rename(
    columns={'ring_id': 'device_ring_id', 'ring_size': 'device_ring_size'}
).to_parquet(f'{project_dir}/data/device_ring_assignments.parquet')

print("saved.")

saved.


In [49]:
ring_fraud_rates = df_dev[df_dev['ring_size'] >= 3].groupby('ring_id')['isFraud'].mean()
print(ring_fraud_rates.describe())
print("rings with 100% internal fraud rate:", (ring_fraud_rates == 1.0).sum())
print("rings with 0% internal fraud rate:", (ring_fraud_rates == 0.0).sum())

count    5143.000000
mean        0.091598
std         0.246459
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: isFraud, dtype: float64
rings with 100% internal fraud rate: 220
rings with 0% internal fraud rate: 4157
